# DeepClean: Automated Data Engineering Pipeline
### *A clean, scalable approach to data preprocessing and feature engineering.*

---

## 1. Project Overview
DeepClean is designed to solve the common "messy data" problem in data engineering. Instead of writing ad-hoc scripts for every dataset, DeepClean provides a reproducible class-based architecture to standardize:
* **Integrity**: Duplicate removal.
* **Feature Engineering**: Automated date part extraction.
* **Data Quality**: Imputation of missing values.
* **Robustness**: Outlier clipping via Z-score.
* **Model Readiness**: Categorical encoding and feature scaling.

## 2. Environment Setup
Initializing the core stack: `pandas` for orchestration, `scikit-learn` for algorithmic cleaning, and `scipy` for statistical detection.

In [5]:
# Standard Data Stack
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import gc

# Machine Learning Utilities
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer

print("Libraries imported successfully!")

Libraries imported successfully!


## 3. The Core Engine: `DeepClean` Class
The class follows a **Fluent Interface** pattern, where each method returns the updated object/dataframe and logs its changes to a central report.

In [6]:
class DeepClean:
    def __init__(self, df):
        """Initialize the cleaning tool with a copy of the dataframe to preserve original data."""
        self.df = df.copy()
        self.report = {}

    def remove_duplicates(self):
        """Removes exact duplicate rows."""
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        self.report['duplicates_removed'] = before - len(self.df)
        return self.df

    def process_dates(self, date_columns=None):
        """Extracts year, month, day, and dayofweek features from timestamp columns."""
        # Identify potential date columns
        cols = date_columns if date_columns else self.df.select_dtypes(include=['datetime64', 'object']).columns
        processed_count = 0
        
        for col in cols:
            try:
                # Only proceed if the column is already datetime or can be converted with at least some valid results
                temp_dates = pd.to_datetime(self.df[col], errors='coerce')
                
                # Check if we actually have valid dates (not all NaT)
                if temp_dates.notnull().any():
                    self.df[col] = temp_dates
                    self.df[f'{col}_year'] = self.df[col].dt.year
                    self.df[f'{col}_month'] = self.df[col].dt.month
                    self.df[f'{col}_day'] = self.df[col].dt.day
                    self.df[f'{col}_dayofweek'] = self.df[col].dt.dayofweek
                    processed_count += 1
            except Exception:
                continue
                
        self.report['date_columns_processed'] = processed_count
        return self.df

    def handle_missing_values(self, strategy='mean', columns=None):
        """Imputes missing values in numeric columns while handling empty features."""
        cols_to_fix = columns if columns else self.df.select_dtypes(include=[np.number]).columns
        
        if len(cols_to_fix) == 0:
            self.report['missing_values_filled'] = 0
            return self.df
            
        missing_before = self.df[cols_to_fix].isnull().sum().sum()
        
        # Use keep_empty_features=True if available, otherwise handle shape manually
        try:
            imputer = SimpleImputer(strategy=strategy, keep_empty_features=True)
            self.df[cols_to_fix] = imputer.fit_transform(self.df[cols_to_fix])
        except TypeError: # Older sklearn version doesn't support keep_empty_features
            imputer = SimpleImputer(strategy=strategy)
            # Only impute columns that have at least one non-null value to avoid shape mismatch
            valid_cols = [c for c in cols_to_fix if self.df[c].notnull().any()]
            if valid_cols:
                self.df[valid_cols] = imputer.fit_transform(self.df[valid_cols])
        
        self.report['missing_values_filled'] = int(missing_before)
        return self.df

    def handle_outliers(self, threshold=3.0):
        """Clips outliers using Z-score statistics."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        outliers_count = 0
        
        for col in numeric_cols:
            if self.df[col].std() == 0 or self.df[col].isnull().all(): continue
            z_scores = np.abs(stats.zscore(self.df[col].dropna()))
            outlier_mask = z_scores > threshold
            outliers_count += outlier_mask.sum()
            
            upper = self.df[col].mean() + threshold * self.df[col].std()
            lower = self.df[col].mean() - threshold * self.df[col].std()
            self.df[col] = self.df[col].clip(lower=lower, upper=upper)
            
        self.report['outliers_clipped'] = int(outliers_count)
        return self.df

    def encode_categorical(self):
        """Converts object columns to numeric labels."""
        obj_cols = self.df.select_dtypes(include=['object']).columns
        le = LabelEncoder()
        
        for col in obj_cols:
            self.df[col] = le.fit_transform(self.df[col].astype(str))
            
        self.report['encoded_columns_count'] = len(obj_cols)
        return self.df

    def normalize_data(self, method='standard'):
        """Scales numeric features for model training."""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) == 0:
            self.report['normalization_method'] = 'none'
            return self.df
            
        scaler = StandardScaler() if method == 'standard' else MinMaxScaler()
        self.df[numeric_cols] = scaler.fit_transform(self.df[numeric_cols])
        self.report['normalization_method'] = method
        return self.df

    def clean(self):
        """The Orchestrator: Runs the full pipeline in sequence."""
        print("--- Executing Automated Cleaning Pipeline ---")
        self.remove_duplicates()
        self.process_dates()
        self.handle_missing_values()
        self.handle_outliers()
        self.encode_categorical()
        self.normalize_data()
        print("--- Pipeline Complete ---")
        return self.df

    def get_summary(self):
        return self.report

## 4. Synthetic Data Generation
Generating a messy dataset with 10,000 rows to simulate real-world data engineering constraints.

In [7]:
def generate_messy_data(n_rows=10000):
    np.random.seed(42)
    
    # Generate base features
    df = pd.DataFrame({
        'age': np.random.normal(35, 12, n_rows).astype(int),
        'income': np.random.lognormal(10, 1, n_rows),
        'city': np.random.choice(['New York', 'London', 'Paris', 'Tokyo'], n_rows),
        'timestamp': pd.to_datetime('2023-01-01') + pd.to_timedelta(np.random.randint(0, 365, n_rows), unit='D'),
        'notes': np.random.choice(['A', 'B', 'C', None], n_rows)
    })

    # Inject Anomalies
    df.loc[df.sample(frac=0.1).index, 'age'] = np.nan
    df.loc[df.sample(frac=0.01).index, 'age'] = 999  # Outlier
    
    # Duplicates
    df = pd.concat([df, df.sample(frac=0.05)], ignore_index=True)
    
    return df

df = generate_messy_data(10000)
print(f"Dataset generated. Shape: {df.shape}")

Dataset generated. Shape: (10500, 5)


## 5. Automated Cleaning Execution
Applying the `DeepClean` pipeline and reviewing the transformation report.

In [8]:
cleaner = DeepClean(df)
clean_df = cleaner.clean()

print("\nTransformation Summary:")
display(cleaner.get_summary())

print("\nHead of Cleaned Data:")
display(clean_df.head())

--- Executing Automated Cleaning Pipeline ---
--- Pipeline Complete ---

Transformation Summary:


C:\Users\HELAL\AppData\Local\Temp\ipykernel_25236\3940898649.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_dates = pd.to_datetime(self.df[col], errors='coerce')
C:\Users\HELAL\AppData\Local\Temp\ipykernel_25236\3940898649.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_dates = pd.to_datetime(self.df[col], errors='coerce')


{'duplicates_removed': 500,
 'date_columns_processed': 1,
 'missing_values_filled': 991,
 'outliers_clipped': 259,
 'encoded_columns_count': 2,
 'normalization_method': 'standard'}


Head of Cleaned Data:


,age,income,city,timestamp,notes,timestamp_year,timestamp_month,timestamp_day,timestamp_dayofweek
0,0.046808,-0.644232,1.346921,2023-01-10,-1.332127,0.0,-1.610342,-0.651804,-1.010177
1,-0.171780,-0.508750,-0.441106,2023-11-04,-0.438382,0.0,1.296727,-1.338312,0.988591
2,0.109261,-0.618910,-1.335120,2023-02-14,-0.438382,0.0,-1.319635,-0.194133,-1.010177
3,0.452757,-0.284306,-0.441106,2023-09-06,1.349108,0.0,0.715313,-1.109476,-0.510485
4,-0.203007,1.011610,1.346921,2023-01-30,1.349108,0.0,-1.610342,1.636553,-1.509869


## 6. Final Data Quality Assessment
Verifying that the pipeline achieved its goals: zero missing values, zero duplicates, and encoded/scaled features.

In [9]:
print("Missing Values:", clean_df.isnull().sum().sum())
print("Duplicates:", clean_df.duplicated().sum())
print("Numeric Datatypes only:", clean_df.select_dtypes(include=[np.number]).shape[1] == clean_df.shape[1])
clean_df.info()

Missing Values: 0
Duplicates: 0
Numeric Datatypes only: False
<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   age                  10000 non-null  float64       
 1   income               10000 non-null  float64       
 2   city                 10000 non-null  float64       
 3   timestamp            10000 non-null  datetime64[ns]
 4   notes                10000 non-null  float64       
 5   timestamp_year       10000 non-null  float64       
 6   timestamp_month      10000 non-null  float64       
 7   timestamp_day        10000 non-null  float64       
 8   timestamp_dayofweek  10000 non-null  float64       
dtypes: datetime64[ns](1), float64(8)
memory usage: 781.2 KB
